# ARTEMIS: A Neuro-Symbolic Framework for Economically Constrained Market Dynamics
**ArXivist-generated reproduction notebook**

Paper: [arXiv:2603.18107](https://arxiv.org/abs/2603.18107) — Rahul D Ray (2026), BITS Pilani

Generated: 2026-07-18

## ⚠ Human review required
This paper's SIR confidence is **0.64** (below the 0.65 auto-proceed threshold). Core
hyperparameters (latent dimension, SDE step count, all loss weights, symbolic basis library)
are never given numeric values anywhere in the paper. Two paper-internal inconsistencies were
found (an unreported "6th baseline"; a DSLOB feature-count discrepancy), plus a third noticed
during code generation (Time-IMM directional accuracy: 96.0% in the abstract vs. 86.0% in
Table 2). The paper's own Appendix A.4.1 also concedes its Feynman-Kac PDE regularizer is
theoretically incomplete. See `sir-registry/arxiv_2603_18107/sir.json` for full detail.

This notebook walks through ARTEMIS's four modules with toy inputs, runs a small end-to-end
fit cycle on a **placeholder synthetic dataset** (the paper's real DSLOB seed data is unnamed
and unavailable — see `data/README_data.md`), and shows how your results compare to the
paper's reported numbers.


In [ ]:
# Environment check.
import sys
print(f"Python: {sys.version}")

try:
    import torch
    print(f"torch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if not torch.cuda.is_available():
        print("No GPU detected -- this notebook will run on CPU (slower, but the small-scale "
              "demo cells below are sized to complete in a few minutes on CPU).")
except ImportError:
    print("torch: NOT INSTALLED -- run the install cell below first.")

for pkg in ["numpy", "pandas", "scipy", "matplotlib", "sklearn", "yaml"]:
    try:
        mod = __import__(pkg)
        print(f"{pkg}: OK ({getattr(mod, '__version__', 'version unknown')})")
    except ImportError as e:
        print(f"{pkg}: MISSING -- {e}")


In [ ]:
# Install the project in editable mode (run once).
import subprocess, sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
result = subprocess.run([sys.executable, "-m", "pip", "install", "-e", project_root],
                          capture_output=True, text=True)
print(result.stdout[-2000:] if result.returncode == 0 else result.stderr[-2000:])


In [ ]:
# Fallback: make the package importable without installing (e.g. if pip install is
# blocked, as in some sandboxed/offline environments).
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print(f"Using project root: {project_root}")


## Paper Overview

**Problem**: deep learning models in quantitative finance are black boxes that ignore
fundamental economic principles like no-arbitrage. ARTEMIS is a neuro-symbolic framework
addressing this with four coupled modules:

1. **Laplace Neural Operator** (Module 1) — continuous-time encoder for irregularly sampled
   market data, using a learnable Laplace-domain kernel: no interpolation needed.
2. **Neural SDE** (Module 2) — drift/diffusion networks evolving a latent state
   $dz = \mu_\theta(z,t)dt + \sigma_\phi(z,t)dW$, regularised by two physics-informed losses:
   - **Feynman-Kac PDE residual** $\mathcal{L}_{PDE}$ — enforces local no-arbitrage via an
     auxiliary pricing network. *(The paper's own Appendix A.4.1 admits this doesn't fully
     enforce no-arbitrage in the strict theoretical sense — it's an empirically useful
     regularizer, not a proof.)*
   - **Market-price-of-risk penalty** $\mathcal{L}_{MPR}$ — bounds the instantaneous Sharpe ratio.
3. **Symbolic bottleneck** (Module 3) — distills latent dynamics into a sparse, interpretable
   combination of basis functions (moving averages, ratios, differences, variances) via a
   two-phase teacher-student procedure.
4. **Conformal allocation** (Module 4) — adaptive conformal prediction intervals, plus an
   optional differentiable Kelly-criterion portfolio layer.

**Key empirical findings** (Table 2/3):
- ARTEMIS achieves the best directional accuracy on DSLOB (64.96%) and Time-IMM (per Table 2,
  86.0% — see the abstract/Table-2 discrepancy noted above).
- Ablation: removing the PDE loss collapses DSLOB directional accuracy from 64.89% to 50.32%
  (near-random); removing both physics losses drops it to 41.77% (worse than random).
- ARTEMIS underperforms on Optiver (negative RankIC, DirAcc below most baselines), attributed
  to long sequence length, a volatility-focused (not directional) target, and few features.

**Section -> code mapping**:

| Paper section | Code module |
|---|---|
| Sec. 4.1 (Laplace Neural Operator) | `src/arxivist_artemis/models/laplace_neural_operator.py` |
| Sec. 4.2 (Neural SDE) | `src/arxivist_artemis/models/neural_sde.py` |
| Sec. 4.2.1-4.2.2 (physics losses) | `src/arxivist_artemis/models/physics_losses.py` |
| Sec. 4.3 (symbolic bottleneck) | `src/arxivist_artemis/models/symbolic_bottleneck.py` |
| Sec. 4.4 (conformal allocation) | `src/arxivist_artemis/models/conformal_allocation.py` |
| Sec. 4.6, Algorithm 1 (full model + training) | `src/arxivist_artemis/models/artemis.py` |
| Sec. 3.4 (DSLOB synthetic data) | `src/arxivist_artemis/data/preprocessing.py::DSLOBGenerator` |


## Component 1 — Laplace Neural Operator (Section 4.1)

Maps irregularly sampled observations to a continuous latent function via a learnable
Laplace-domain kernel:
$$z(t) = \int_0^T \kappa(t-s)x(s)\,ds + b(t), \qquad \kappa(t) = \sum_k A_k e^{\lambda_k t}$$


In [ ]:
try:
    import torch
    from arxivist_artemis.models.laplace_neural_operator import LaplaceNeuralOperator

    torch.manual_seed(0)
    d_x, d_z = 10, 64  # d_z=64 is ASSUMED (Figure 3 caption hint, SIR confidence 0.35)
    lno = LaplaceNeuralOperator(d_x=d_x, d_z=d_z, n_poles=16, n_fourier=8)

    B, N, M = 4, 20, 20  # batch, observed points, eval grid points
    x = torch.randn(B, N, d_x)
    t = torch.linspace(0, 1, N).unsqueeze(0).expand(B, -1)
    eval_times = torch.linspace(0, 1, M)

    z = lno(x, t, eval_times)
    print(f"Input shape: {x.shape}, observed times: {t.shape}")
    print(f"Output latent function shape: {z.shape}  (expected: [{B}, {M}, {d_z}])")
except Exception as e:
    print(f"[Component 1 failed] {type(e).__name__}: {e}")


## Component 2 — Neural SDE (Section 4.2)

Evolves the latent state via Euler-Maruyama simulation of
$dz = \mu_\theta(z,t)dt + \sigma_\phi(z,t)dW$, with $\sigma_\phi = L_\phi D_\phi$
(Cholesky-style, guaranteeing invertibility).


In [ ]:
try:
    from arxivist_artemis.models.neural_sde import NeuralSDE, DriftNet, DiffusionNet

    d_w = 16  # ASSUMED, unspecified in paper
    drift = DriftNet(d_z, hidden_dim=128)
    diffusion = DiffusionNet(d_z, d_w, hidden_dim=128)
    sde = NeuralSDE(d_z, d_w, drift, diffusion)

    z0 = z[:, 0, :]  # initial latent state from the LNO encoder, from Component 1
    n_steps = 100  # ASSUMED=100 per Figure 2 caption (SIR confidence 0.3)
    z_traj = sde.simulate(z0, n_steps=n_steps, dt=1.0 / n_steps)
    print(f"Initial state shape: {z0.shape}")
    print(f"Simulated trajectory shape: {z_traj.shape}  (expected: [{B}, {n_steps+1}, {d_z}])")
except Exception as e:
    print(f"[Component 2 failed] {type(e).__name__}: {e}")


## Component 3 — Physics-Informed Losses (Section 4.2.1-4.2.2)

The Feynman-Kac PDE residual (via nested `torch.autograd.grad` calls for the Hessian) and the
market-price-of-risk hinge penalty.


In [ ]:
try:
    from arxivist_artemis.models.physics_losses import PhysicsLosses, AuxiliaryPricingNet

    aux_net = AuxiliaryPricingNet(d_z, hidden_dim=128)
    n_coll = 16
    z_coll = z_traj[0, :n_coll, :]  # collocation points along one trajectory
    t_coll = torch.linspace(0, 1, n_coll)

    residuals = PhysicsLosses.feynman_kac_residual(aux_net, drift, diffusion, z_coll, t_coll)
    l_pde = PhysicsLosses.pde_loss(residuals)
    print(f"PDE residuals shape: {residuals.shape}, L_PDE = {l_pde.item():.6f}")

    # NOTE: market_price_of_risk (Eq. 8) is defined as an element-wise ratio
    # lambda(t) = mu(t)/sigma(t), which only makes sense when sigma is square
    # (d_w == d_z). Component 2's demo used d_w=16 != d_z=64 to show the
    # general (non-square) SDE case, so here we build a fresh square
    # DiffusionNet (d_w=d_z) specifically for this MPR demonstration -- this
    # mirrors how ARTEMIS itself is actually configured in practice (the
    # architecture plan documents d_w defaulting to d_z-compatible sizing
    # precisely so the MPR loss is well-defined).
    from arxivist_artemis.models.neural_sde import DiffusionNet
    square_diffusion = DiffusionNet(d_z, d_w=d_z, hidden_dim=128)

    mu_coll = drift(z_coll, t_coll)
    sigma_coll = square_diffusion(z_coll, t_coll)  # [n_coll, d_z, d_z], now square
    lam = PhysicsLosses.market_price_of_risk(mu_coll, sigma_coll)
    l_mpr = PhysicsLosses.mpr_loss(lam, kappa=2.0)
    print(f"Market price of risk shape: {lam.shape}, L_MPR = {l_mpr.item():.6f}")
except Exception as e:
    print(f"[Component 3 failed] {type(e).__name__}: {e}")


## Component 4 — Symbolic Bottleneck (Section 4.3)

Distills predictions into a sparse combination of interpretable basis functions
(moving averages, ratios, differences, variances) computed on the **raw input window**.


In [ ]:
try:
    from arxivist_artemis.models.symbolic_bottleneck import SymbolicBottleneck, BasisLibrary

    n_channels = d_x
    basis_lib = BasisLibrary(n_channels, lags=[2, 5, 10, 20])
    symb = SymbolicBottleneck(basis_lib.n_basis)

    x_window = torch.randn(B, N, n_channels)
    basis_features = basis_lib.compute_features(x_window)
    y_symb = symb(basis_features, tau=1.0)
    print(f"Basis library size: {basis_lib.n_basis}")
    print(f"Basis features shape: {basis_features.shape}")
    print(f"Symbolic prediction shape: {y_symb.shape}")

    # export_formula() addresses a gap in the paper itself -- no example distilled
    # formula is ever shown despite interpretability being a headline contribution.
    print(symb.export_formula(basis_lib.feature_names(), top_k=5))
except Exception as e:
    print(f"[Component 4 failed] {type(e).__name__}: {e}")


## Component 5 — Full ARTEMIS Assembly (Section 4.6, Algorithm 1)

Assembles all four modules, runs the Phase-1 pretraining forward pass, and computes the
composite loss $\mathcal{L}_{total} = \mathcal{L}_{forecast} + \lambda_1\mathcal{L}_{PDE} +
\lambda_2\mathcal{L}_{MPR} + \lambda_3\mathcal{L}_{consist}$.


In [ ]:
try:
    from arxivist_artemis.models.artemis import ARTEMIS

    model = ARTEMIS(d_x=d_x, d_z=d_z, d_w=d_w, n_sde_steps=20)  # small n_sde_steps for a quick demo
    n_params = sum(p.numel() for p in model.parameters())
    print(f"ARTEMIS built with {n_params:,} parameters (toy dims: d_x={d_x}, d_z={d_z}, d_w={d_w})")

    eval_times_full = torch.linspace(0, 1, N)
    t_obs_full = eval_times_full.unsqueeze(0).expand(B, -1)
    out = model.forward_pretrain(x, t_obs_full, eval_times_full)
    print(f"y_hat shape: {out['y_hat'].shape}, z_traj shape: {out['z_traj'].shape}")

    y_true_toy = torch.randn(B)
    lambdas = {"pde": 1.0, "mpr": 1.0, "consist": 1.0}  # ASSUMED -- paper gives no numeric values
    losses = model.compute_losses(out["y_hat"], y_true_toy, out["z_traj"], out["z_enc"], lambdas)
    for k, v in losses.items():
        print(f"  {k}: {v.item():.6f}")
except Exception as e:
    print(f"[Component 5 failed] {type(e).__name__}: {e}")


## Mini End-to-End Run (placeholder synthetic DSLOB data)

The paper's real DSLOB dataset is generated from an unnamed "real high-frequency limit order
book dataset" (Section 3.4) that is never identified — see `data/README_data.md`. The cells
below generate a **placeholder** seed series (`data/make_synthetic_seed_lob.py`), run it
through the actual DSLOB *generation procedure* (crash detection, Vasicek/GARCH fitting +
amplification, VAR(1)-correlated noise, time warping), and train a small ARTEMIS model on it.

This validates that the pipeline *runs correctly* — it does **not** reproduce the paper's
actual reported numbers (Table 2/3), since the true seed data is unavailable.


In [ ]:
# 1) Generate placeholder seed data + run the DSLOB generation pipeline.
import subprocess, sys, os

seed_script = os.path.join(project_root, "data", "make_synthetic_seed_lob.py")
result = subprocess.run([sys.executable, seed_script, "--out",
                          os.path.join(project_root, "data", "raw", "seed_lob.npz"),
                          "--n-obs", "2000", "--seed", "123"],
                          capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)

prepare_script = os.path.join(project_root, "scripts", "prepare_data.py")
result = subprocess.run([sys.executable, prepare_script, "--dataset", "dslob",
                          "--raw-dir", os.path.join(project_root, "data", "raw"),
                          "--out-dir", os.path.join(project_root, "data", "processed"),
                          "--n-samples", "300", "--seed", "123"],
                          capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)


In [ ]:
# 2) Train a small ARTEMIS model for a couple of epochs (CPU-friendly demo scale).
import numpy as np

data_path = os.path.join(project_root, "data", "processed", "dslob.npz")
blob = np.load(data_path)
X_demo = torch.tensor(blob["X"][:64], dtype=torch.float32)  # small subset for a quick demo
y_demo = torch.tensor(blob["y"][:64], dtype=torch.float32)
print(f"Demo dataset: X={X_demo.shape}, y={y_demo.shape}")

demo_model = ARTEMIS(d_x=X_demo.shape[-1], d_z=32, d_w=16, n_sde_steps=20,
                      use_symbolic=True, use_conformal=False)
optimizer = torch.optim.Adam(demo_model.parameters(), lr=1e-3)

for epoch in range(3):
    optimizer.zero_grad()
    eval_times = torch.linspace(0, 1, X_demo.shape[1])
    t_obs = eval_times.unsqueeze(0).expand(X_demo.shape[0], -1)
    out = demo_model.forward_pretrain(X_demo, t_obs, eval_times)
    losses = demo_model.compute_losses(out["y_hat"], y_demo, out["z_traj"], out["z_enc"],
                                        {"pde": 1.0, "mpr": 1.0, "consist": 1.0})
    losses["total"].backward()
    optimizer.step()
    print(f"epoch {epoch+1}: total_loss={losses['total'].item():.4f}, "
          f"forecast={losses['forecast'].item():.4f}, pde={losses['pde'].item():.4f}")


## Paper Results Comparison

The table below shows the paper's actually reported results (from `sir.json ->
evaluation_protocol.reported_results`), computed on the paper's real (proprietary/synthetic)
data — **not** the tiny placeholder demo above, which is for pipeline sanity-checking only.


In [ ]:
paper_results_dslob_table2 = {
    "RMSE": 0.03615, "RankIC": 0.08791, "DirAcc": 0.6496,
}
paper_ablation_diracc = {
    "A0_Full": 0.6489, "A1_NoSDE": 0.6459, "A2_NoPDE": 0.5032,
    "A3_NoMPR": 0.5682, "A4_NoPhysics": 0.4177, "A5_NoConsistency": 0.3754, "A6_MLP": 0.3504,
}
print("Paper Table 2 (DSLOB, ARTEMIS):")
for k, v in paper_results_dslob_table2.items():
    print(f"  {k}: {v}")
print("\nPaper Table 3 (DSLOB ablation, Directional Accuracy):")
for k, v in paper_ablation_diracc.items():
    print(f"  {k}: {v:.4f}")

print("\nTo reproduce these exact numbers you need the paper's actual DSLOB data (unavailable --")
print("see data/README_data.md) or the Jane Street / Optiver / Time-IMM public datasets, then:")
print("  python train.py --config configs/config.yaml --model artemis --dataset <dataset>")
print("  python scripts/run_ablation.py --config configs/config.yaml --variant <variant>")
print("Then use the Results Comparator (Stage 6) to compare your outputs against these targets.")


## What to do next

1. **Data**: Jane Street / Optiver / Time-IMM are public (see `data/README_data.md` for links);
   DSLOB's true seed dataset is unnamed and unavailable — only the *generation procedure* can
   be exercised, via a placeholder seed series.
2. **Full training**: `python train.py --config configs/config.yaml --model artemis --dataset <dataset>`
3. **Ablation study**: `python scripts/run_ablation.py --config configs/config.yaml --variant <A0_Full..A6_MLP>`
4. **Evaluate**: `python evaluate.py --config configs/config.yaml --checkpoint <ckpt> --dataset <dataset>`
5. **Figures**: `python scripts/plot_figures.py --checkpoint <ckpt> --dataset <dataset>`
6. **Compare results**: feed your `results/metrics_report.md` back to ArXivist's Results Comparator (Stage 6).

**Top implementation assumptions to be aware of** (see `sir.json -> implementation_assumptions`
for the full list, and `README.md -> Known Limitations`):

1. *Latent dimension d_z=64* — inferred only from a figure caption; SIR confidence **0.35**.
2. *SDE step count / all four loss weights* — never given numeric values; SIR confidence **0.3**.
3. *Feynman-Kac PDE regularizer's theoretical completeness* — the paper's own Appendix A.4.1
   admits this does not provably enforce no-arbitrage; it's an empirically useful but
   theoretically incomplete regularizer.
4. *DSLOB's seed dataset* — unnamed in the paper; cannot be obtained, only its generation
   *procedure* can be exercised on a placeholder.

See `notebooks/explore_arxiv_2603_18107.ipynb` for visual exploration of drift/diffusion
profiles, PCA latent vector fields, and the ablation study's directional-accuracy comparison.
